# nanoGPT

> **Update Nov 2025**: nanoGPT has a new and improved cousin called [nanochat](https://github.com/karpathy/nanochat). It is very likely you meant to use/find nanochat instead. nanoGPT (this repo) is now very old and deprecated but I will leave it up for posterity.

The simplest, fastest repository for training/finetuning medium-sized GPTs. It is a rewrite of [minGPT](https://github.com/karpathy/minGPT) that prioritizes teeth over education. Still under active development, but currently the file `train.py` reproduces GPT-2 (124M) on OpenWebText, running on a single 8XA100 40GB node in about 4 days of training. The code itself is plain and readable: `train.py` is a ~300-line boilerplate training loop and `model.py` a ~300-line GPT model definition, which can optionally load the GPT-2 weights from OpenAI. That's it.

Because the code is so simple, it is very easy to hack to your needs, train new models from scratch, or finetune pretrained checkpoints (e.g. biggest one currently available as a starting point would be the GPT-2 1.3B model from OpenAI).

---
**笔记**



## Install

```
pip install torch numpy transformers datasets tiktoken wandb tqdm
```

Dependencies:

- [pytorch](https://pytorch.org)
- [numpy](https://numpy.org/install/)
- `transformers` for huggingface transformers (to load GPT-2 checkpoints)
- `datasets` for huggingface datasets (if you want to download + preprocess OpenWebText)
- `tiktoken` for OpenAI's fast BPE code
- `wandb` for optional logging
- `tqdm` for progress bars

In [ ]:
# !pip install torch numpy transformers datasets tiktoken wandb tqdm

---
**笔记**



## Quick Start

If you are not a deep learning professional and you just want to feel the magic and get your feet wet, the fastest way to get started is to train a character-level GPT on the works of Shakespeare. First, we download it as a single (1MB) file and turn it from raw text into one large stream of integers:

In [ ]:
# !python data/shakespeare_char/prepare.py

This creates a `train.bin` and `val.bin` in that data directory. Now it is time to train your GPT. The size of it very much depends on the computational resources of your system.

### I have a GPU

Great, we can quickly train a baby GPT with the settings provided in the [config/train_shakespeare_char.py](config/train_shakespeare_char.py) config file:

In [ ]:
# !python train.py config/train_shakespeare_char.py


step 5000: train loss 0.6281, val loss 1.7000
iter 5000: loss 0.8324, time 2716.35ms, mfu 18.38%


---
**笔记：MFU（Model FLOPS Utilization）**

输出中的 `mfu 18.38%` 是 **模型浮点运算利用率**，衡量实际用了多少比例的 GPU 算力。

$$\text{MFU} = \frac{\text{实际吞吐量（FLOP/s）}}{\text{硬件峰值算力（FLOP/s）}}$$

**计算公式：** 对于 Transformer，每步理论 FLOPs ≈ $6 \times N \times T$（N=参数量，T=token数，系数6=前向2+反向4）

| MFU 值 | 含义 |
|--------|------|
| 50%+ | 优秀，工程优化到位 |
| 30–40% | 一般水平 |
| <20% | 存在瓶颈（IO、通信、显存带宽等）|

这里 18.38% 属于偏低，因为是在单 GPU 上训练小模型，overhead 比例更大。PaLM 论文报告的 MFU 约为 46.2%。


If you peek inside it, you'll see that we're training a GPT with a context size of up to 256 characters, 384 feature channels, and it is a 6-layer Transformer with 6 heads in each layer. On one A100 GPU this training run takes about 3 minutes and the best validation loss is 1.4697. Based on the configuration, the model checkpoints are being written into the `--out_dir` directory `out-shakespeare-char`. So once the training finishes we can sample from the best model by pointing the sampling script at this directory:

---
**笔记：训练显存估算公式**

```
python train.py config/train_shakespeare_char.py --batch_size=1536
```

训练显存由四部分构成：

$$\text{显存} = \underbrace{\text{参数}}_{\text{极小}} + \underbrace{\text{梯度+优化器}}_{\text{极小}} + \underbrace{\text{激活值}}_{\textbf{主体}}$$

---

### 一、参数 + 优化器状态（混合精度 AMP）

| 组成 | 精度 | 字节/参数 |
|------|------|-----------|
| bf16 权重 | bf16 | 2 |
| fp32 主权重 | fp32 | 4 |
| fp32 梯度 | fp32 | 4 |
| Adam 一阶矩 m | fp32 | 4 |
| Adam 二阶矩 v | fp32 | 4 |
| **合计** | | **≈ 18 bytes/param** |

本模型参数量（n_layer=6, n_head=6, n_embd=384, vocab_size=65）：

$$N \approx 6 \times \underbrace{(384 \times 1152 + 384^2 + 384 \times 1536 \times 2)}_{\text{每层 Attn+MLP}} + \text{embedding} \approx 10.77\text{M}$$

$$\text{参数+优化器} = 10.77\text{M} \times 18 \approx \mathbf{0.19\text{ GB}} \quad \text{（可忽略）}$$

---

### 二、激活值（主体）—— 以 batch\_size=1536 为例

> 反向传播需保留前向所有中间张量，不用 gradient checkpointing 时全部常驻显存。

**符号：** B=1536, T=256, d=384, 4d=1536, bf16=2 bytes

**每个 Transformer Block 需保留的张量：**

| 张量 | Shape | 大小 |
|------|-------|------|
| LN1 输入（残差流） | B×T×d | 302 MB |
| c\_attn 输入（算权重梯度用） | B×T×d | 302 MB |
| Q, K, V（Flash Attn backward 用） | 3 × B×T×d | 906 MB |
| Attention 输出（算 c\_proj 权重梯度） | B×T×d | 302 MB |
| 第一个残差后（LN2 输入） | B×T×d | 302 MB |
| c\_fc 输入（算 c\_fc 权重梯度） | B×T×d | 302 MB |
| GELU 输入（算 GELU 梯度） | B×T×4d | **1207 MB** |
| GELU 输出（算 c\_proj 权重梯度） | B×T×4d | **1207 MB** |
| MLP 后残差流 | B×T×d | 302 MB |
| **每层小计** | | **≈ 5.3 GB** |

$$\text{6层激活值} = 6 \times 5.3 \approx \mathbf{31.8\text{ GB}}$$

**其他开销：**
- Embedding 输出 + 最终 LN + Logits（B×T×65）≈ 0.65 GB
- CUDA context + PyTorch overhead ≈ 2~3 GB

$$\boxed{\text{总显存} \approx 31.8 + 0.65 + 0.19 + 3 \approx \mathbf{36\text{ GB}}}$$

---

### 三、关键结论

MLP 的两个大矩阵（GELU前后各 B×T×4d）是**最大的单块张量**，每层各占 1.2 GB，六层共占 **14.5 GB**，超过总显存的 40%。

减少显存的直接手段：
- **gradient checkpointing**：重算激活值而不存储，显存降至约 $O(\sqrt{L})$，但增加约 33% 计算量
- **减小 batch\_size** 或 **block\_size**：激活值与两者线性相关

In [ ]:
# !python train.py config/train_shakespeare_char.py --batch_size=1536


In [ ]:
# !python sample.py --out_dir=out-shakespeare-char

This generates a few samples, for example:

```
ANGELO:
And cowards it be strawn to my bed,
And thrust the gates of my threats,
Because he that ale away, and hang'd
An one with him.

DUKE VINCENTIO:
I thank your eyes against it.

DUKE VINCENTIO:
Then will answer him to save the malm:
And what have you tyrannous shall do this?
```

Not bad for a character-level model after 3 minutes of training on a GPU.

### I only have a macbook (or other cheap computer)

No worries, we can still train a GPT but we want to dial things down a notch.

In [ ]:
# CPU
# !python train.py config/train_shakespeare_char.py --device=cpu --compile=False --eval_iters=20 --log_interval=1 --block_size=64 --batch_size=12 --n_layer=4 --n_head=4 --n_embd=128 --max_iters=2000 --lr_decay_iters=2000 --dropout=0.0

Here, since we are running on CPU instead of GPU we must set both `--device=cpu` and also turn off PyTorch 2.0 compile with `--compile=False`. Then when we evaluate we get a bit more noisy but faster estimate (`--eval_iters=20`, down from 200), our context size is only 64 characters instead of 256, and the batch size only 12 examples per iteration, not 64. We'll also use a much smaller Transformer (4 layers, 4 heads, 128 embedding size), and decrease the number of iterations to 2000. This still runs in about ~3 minutes, but gets us a loss of only 1.88.

In [ ]:
# !python sample.py --out_dir=out-shakespeare-char --device=cpu

Finally, on Apple Silicon Macbooks and with a recent PyTorch version make sure to add `--device=mps` (short for "Metal Performance Shaders"); PyTorch then uses the on-chip GPU that can *significantly* accelerate training (2-3X) and allow you to use larger networks.

---
**笔记**



## Reproducing GPT-2

A more serious deep learning professional may be more interested in reproducing GPT-2 results. So here we go - we first tokenize the dataset, in this case the [OpenWebText](https://openwebtext2.readthedocs.io/en/latest/), an open reproduction of OpenAI's (private) WebText:

In [ ]:
# !python data/openwebtext/prepare.py

This downloads and tokenizes the [OpenWebText](https://huggingface.co/datasets/openwebtext) dataset. It will create a `train.bin` and `val.bin` which holds the GPT2 BPE token ids in one sequence, stored as raw uint16 bytes. Then we're ready to kick off training. To reproduce GPT-2 (124M) you'll want at least an 8X A100 40GB node and run:

In [ ]:
# 8-GPU single node
# !torchrun --standalone --nproc_per_node=8 train.py config/train_gpt2.py

This will run for about 4 days using PyTorch Distributed Data Parallel (DDP) and go down to loss of ~2.85. Now, a GPT-2 model just evaluated on OWT gets a val loss of about 3.11, but if you finetune it it will come down to ~2.85 territory (due to an apparent domain gap), making the two models ~match.

If you're in a cluster environment and you are blessed with multiple GPU nodes you can make GPU go brrrr e.g. across 2 nodes like:

In [ ]:
# Multi-node example (2 nodes, 8 GPUs each)
# Run on the first (master) node with example IP 123.456.123.456:
# !torchrun --nproc_per_node=8 --nnodes=2 --node_rank=0 --master_addr=123.456.123.456 --master_port=1234 train.py
# Run on the worker node:
# !torchrun --nproc_per_node=8 --nnodes=2 --node_rank=1 --master_addr=123.456.123.456 --master_port=1234 train.py

It is a good idea to benchmark your interconnect (e.g. iperf3). In particular, if you don't have Infiniband then also prepend `NCCL_IB_DISABLE=1` to the above launches. Your multinode training will work, but most likely *crawl*. By default checkpoints are periodically written to the `--out_dir`. We can sample from the model by simply `python sample.py`.

---
**笔记**



## Baselines

OpenAI GPT-2 checkpoints allow us to get some baselines in place for openwebtext. We can get the numbers as follows:

In [ ]:
# !python train.py config/eval_gpt2.py
# !python train.py config/eval_gpt2_medium.py
# !python train.py config/eval_gpt2_large.py
# !python train.py config/eval_gpt2_xl.py

Observed losses on train and val:

| model | params | train loss | val loss |
| ------| ------ | ---------- | -------- |
| gpt2 | 124M | 3.11 | 3.12 |
| gpt2-medium | 350M | 2.85 | 2.84 |
| gpt2-large | 774M | 2.66 | 2.67 |
| gpt2-xl | 1558M | 2.56 | 2.54 |

However, we have to note that GPT-2 was trained on (closed, never released) WebText, while OpenWebText is just a best-effort open reproduction of this dataset. This means there is a dataset domain gap. Indeed, taking the GPT-2 (124M) checkpoint and finetuning on OWT directly for a while reaches loss down to ~2.85. This then becomes the more appropriate baseline w.r.t. reproduction.

---
**笔记**



## Finetuning

Finetuning is no different than training, we just make sure to initialize from a pretrained model and train with a smaller learning rate. For an example of how to finetune a GPT on new text go to `data/shakespeare` and run `prepare.py` to download the tiny shakespeare dataset and render it into a `train.bin` and `val.bin`, using the OpenAI BPE tokenizer from GPT-2. Unlike OpenWebText this will run in seconds. Finetuning can take very little time, e.g. on a single GPU just a few minutes. Run an example finetuning like:

In [ ]:
# !python train.py config/finetune_shakespeare.py

This will load the config parameter overrides in `config/finetune_shakespeare.py`. Basically, we initialize from a GPT2 checkpoint with `init_from` and train as normal, except shorter and with a small learning rate. If you're running out of memory try decreasing the model size (they are `{'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}`) or possibly decreasing the `block_size` (context length). The best checkpoint (lowest validation loss) will be in the `out_dir` directory, e.g. in `out-shakespeare` by default, per the config file. You can then run the code in `sample.py --out_dir=out-shakespeare`:

```
THEODORE:
Thou shalt sell me to the highest bidder: if I die,
I sell thee to the first; if I go mad,
I sell thee to the second; if I
lie, I sell thee to the third; if I slay,
I sell thee to the fourth: so buy or sell,
I tell thee again, thou shalt not sell my
possession.
```

---
**笔记**



## Sampling / Inference

Use the script `sample.py` to sample either from pre-trained GPT-2 models released by OpenAI, or from a model you trained yourself. For example, here is a way to sample from the largest available `gpt2-xl` model:

In [ ]:
# !python sample.py \
#     --init_from=gpt2-xl \
#     --start="What is the answer to life, the universe, and everything?" \
#     --num_samples=5 --max_new_tokens=100

If you'd like to sample from a model you trained, use the `--out_dir` to point the code appropriately. You can also prompt the model with some text from a file, e.g. `python sample.py --start=FILE:prompt.txt`.

---
**笔记**



## Efficiency Notes

For simple model benchmarking and profiling, `bench.py` might be useful. It's identical to what happens in the meat of the training loop of `train.py`, but omits much of the other complexities.

In [ ]:
# !python bench.py

Note that the code by default uses [PyTorch 2.0](https://pytorch.org/get-started/pytorch-2.0/). `torch.compile()` is available and makes a noticeable improvement, e.g. cutting down iteration time from ~250ms / iter to 135ms / iter.

---
**笔记**



## Troubleshooting

Note that by default this repo uses PyTorch 2.0 (i.e. `torch.compile`). This is fairly new and experimental, and not yet available on all platforms (e.g. Windows). If you're running into related error messages try to disable this by adding `--compile=False` flag. This will slow down the code but at least it will run.

For some context on this repository, GPT, and language modeling it might be helpful to watch Karpathy's [Zero To Hero series](https://karpathy.ai/zero-to-hero.html). Specifically, the [GPT video](https://www.youtube.com/watch?v=kCc8FmEb1nY) is popular if you have some prior language modeling context.

---
**笔记**



## Todos

- Investigate and add FSDP instead of DDP
- Eval zero-shot perplexities on standard evals (e.g. LAMBADA? HELM? etc.)
- Finetune the finetuning script, I think the hyperparams are not great
- Schedule for linear batch size increase during training
- Incorporate other embeddings (rotary, alibi)
- Separate out the optim buffers from model params in checkpoints I think
- Additional logging around network health (e.g. gradient clip events, magnitudes)
- Few more investigations around better init etc.

## Acknowledgements

All nanoGPT experiments are powered by GPUs on [Lambda labs](https://lambdalabs.com), my favorite Cloud GPU provider. Thank you Lambda labs for sponsoring nanoGPT!